In [ ]:
import sys
print(sys.executable)

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")

# Data Sources

## NOAA

In [ ]:
# Import data from xls file
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pytz


import seaborn as sns

dfd = pd.read_csv('./data/noaa/2023_details.csv')

# For dfd (NOAA data):
# Step 1: Create a combined datetime string
def parse_noaa_datetime(row):
    """
    Parse BEGIN_YEARMONTH, BEGIN_DAY, BEGIN_TIME into a datetime
    """
    year_month = str(int(row['BEGIN_YEARMONTH']))  # e.g., '202301'
    day = str(int(row['BEGIN_DAY'])).zfill(2)  # e.g., '25'
    time_str = str(int(row['BEGIN_TIME'])).zfill(4)  # e.g., '0230' 
    
    # Combine into format: YYYYMMDD HHMM
    datetime_str = f"{year_month}{day} {time_str}"
    
    # Parse to datetime
    return pd.to_datetime(datetime_str, format='%Y%m%d %H%M')

# Apply to dataframe
dfd['datetime'] = dfd.apply(parse_noaa_datetime, axis=1)

# Round to nearest hour to match ERA5 hourly data
dfd['datetime_hourly'] = dfd['datetime'].dt.round('H')

# For dfd - convert to ISO format string
dfd['datetime_iso'] = dfd['datetime_hourly'].dt.strftime('%Y-%m-%dT%H:%M:%S')


# This is the RECOMMENDED approach as it properly handles DST transitions
central_tz = pytz.timezone('America/Chicago')

# Localize the datetime_hourly column to Central timezone
dfd['datetime_central'] = dfd['datetime_hourly'].dt.tz_localize(central_tz, ambiguous='infer', nonexistent='shift_forward')

# Convert to UTC
dfd['datetime_utc'] = dfd['datetime_central'].dt.tz_convert('UTC')

# If you want to remove timezone info but keep the UTC time
dfd['datetime_utc_naive'] = dfd['datetime_utc'].dt.tz_localize(None)

print("Sample conversions:")
print(dfd[['datetime_hourly', 'datetime_utc_naive']].head(10))

# print(dfd[['datetime', 'datetime_hourly', 'datetime_iso']].head(10))


### Data Filtering

Based on:
- Record must have latitude and longitude information
- Selected States
- Selected date and time

#### Remove data with no latitude/longitude (null values)

In [ ]:
# Remove data with no latitude/longitude
dfd = dfd[dfd['BEGIN_LAT'].notnull() | dfd['BEGIN_LON'].notnull()]

#### Selected States and Dates

The first part is focusing on finding the states that has most event of extreme weather, categorized them by weather type. In this case, the categorization will be using heatmap.

In [ ]:
# Create a pivot table: rows=EVENT_TYPE, columns=STATE, values=number of events
heatmap_data = dfd.pivot_table(index='EVENT_TYPE', columns='STATE', aggfunc='size')

plt.figure(figsize=(25, 8))
sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.5, linecolor='gray')
plt.title('Number of Extreme Weather Events by Event Type and State')
plt.xlabel('State')
plt.ylabel('Event Type')
plt.tight_layout()
plt.show()


Based on the heatmap, the selected extreme weather for this research is Thunderstorm Wind in Texas. The state of Georgia and Alabama can also be used.

To reduce the number of data being used, we also be focusing on the month where there has most extreme weather condition.

In [ ]:
# The states that has the most extreme weather events
STATES_LIST = ["TEXAS"]

# Filter dfd to only include rows where STATE is in STATES_LIST
dfd = dfd[dfd['STATE'].isin(STATES_LIST)]

# Filter dfd to only include rows where EVENT_TYPE is 'Thunderstorm Wind'
# dfd = dfd[dfd['EVENT_TYPE'] == 'Thunderstorm Wind']

# Group by month and count the number of Thunderstorm Wind events
grouped_by_month = dfd.groupby('BEGIN_YEARMONTH').size().reset_index(name='event_count')    
print(grouped_by_month)

In [ ]:
# Filter dfd for specific date
specific_date = pd.Timestamp('2023-06-12')  # replace year as appropriate, or set externally
dfd = dfd[dfd['datetime_utc_naive'].dt.month == specific_date.month]

print(dfd.shape)

Getting the min and max latitude of the states that becomes the main focus of the data. This information is useful for selecting specific region in ERA5 dataset.

In [ ]:
# Find the min and max latitude for each state
min_max_latitude = dfd.groupby('STATE')['BEGIN_LAT'].agg(['min', 'max'])
print("Latitude:")
print(min_max_latitude)

print("")

# Find the min and max longitude for each state
min_max_longitude = dfd.groupby('STATE')['BEGIN_LON'].agg(['min', 'max'])
print("Longitude:")
print(min_max_longitude)

In [ ]:
state_event_counts = dfd.groupby('STATE').size().reset_index(name='event_count')
# Focus only on event_type 'Thunderstorm Wind'

state_event_counts_thunderstorm = dfd.groupby('STATE').size().reset_index(name='event_count')
print(state_event_counts_thunderstorm)

# print(state_event_counts)

print(dfd['datetime_utc_naive'])

#### Conform the Latitude and Longitude

Rounding the latitude and longitude in the increment of 0.25. This rounding process will help to conform with the ERA5 dataset in column Time.

In [ ]:
# print("BEFORE ROUNDING:")
# print(dfd[['BEGIN_LAT', 'BEGIN_LON']].head(10))

# Round coordinates to match ERA5 0.25 degree grid
dfd['BEGIN_LAT'] = (dfd['BEGIN_LAT'] / 0.25).round() * 0.25
dfd['BEGIN_LON'] = (dfd['BEGIN_LON'] / 0.25).round() * 0.25
dfd['END_LAT'] = (dfd['END_LAT'] / 0.25).round() * 0.25
dfd['END_LON'] = (dfd['END_LON'] / 0.25).round() * 0.25

# print("AFTER ROUNDING:")
# print(dfd[['BEGIN_LAT', 'BEGIN_LON']].head(10))

# Find the min and max latitude for each state
min_max_latitude = dfd.groupby('STATE')['BEGIN_LAT'].agg(['min', 'max'])
print("Latitude:")
print(min_max_latitude)

print("")

# Find the min and max longitude for each state
min_max_longitude = dfd.groupby('STATE')['BEGIN_LON'].agg(['min', 'max'])
print("Longitude:")
print(min_max_longitude)

### Remove Unnecessary Columns

In [ ]:
columns_to_remove = [
    'TOR_F_SCALE', 
    'TOR_LENGTH', 
    'TOR_WIDTH', 
    'TOR_OTHER_WFO', 
    'TOR_OTHER_WFO_ID', 
    'TOR_OTHER_WFO_NAME', 
    'TOR_OTHER_WFO_STATE',
    'TOR_OTHER_CZ_STATE',
    'TOR_OTHER_CZ_FIPS',
    'TOR_OTHER_CZ_NAME',
    'CZ_FIPS',
    'CZ_NAME',
    'CZ_TYPE',
    'DEATHS_DIRECT', 
    'DEATHS_INDIRECT', 
    'INJURIES_DIRECT', 
    'INJURIES_INDIRECT', 
    'DAMAGE_PROPERTY', 
    'DAMAGE_CROPS', 
    'BEGIN_LOCATION',
    'END_LOCATION',
    'YEAR',
    'MONTH_NAME',
    'BEGIN_YEARMONTH',
    'BEGIN_DAY',
    'BEGIN_TIME',
    'END_YEARMONTH',
    'END_DAY',
    'END_TIME',
    'SOURCE',
    'DATA_SOURCE',
    'WFO',
    'EPISODE_ID',
    'STATE_FIPS',
    'BEGIN_DATE_TIME',
    'END_DATE_TIME',
    'BEGIN_RANGE',
    'END_RANGE',
    'BEGIN_AZIMUTH',
    'END_AZIMUTH',
    'MAGNITUDE',
    'MAGNITUDE_TYPE',    
    'EPISODE_NARRATIVE',
    'EVENT_NARRATIVE',
    'CZ_TIMEZONE',
    'FLOOD_CAUSE',
    'CATEGORY',
    'datetime_hourly',
    'datetime',
    'EVENT_ID',
    'datetime_iso', 'datetime_central', 'datetime_utc', 'datetime_utc_iso']
    
dfd = dfd.drop(columns=columns_to_remove, errors='ignore')


Add column has_extreme_weather as an True/False indicator whether there's a extreme event at the time and location (latitude, longitude).

In [ ]:
# Mark all rows as having extreme weather
dfd['has_extreme_weather'] = True

In [ ]:
# # 1. Define your time range
# start_date = dfd['datetime_utc_naive'].min().floor('H')
# end_date = dfd['datetime_utc_naive'].max().ceil('H')
# hourly_range = pd.date_range(start=start_date, end=end_date, freq='H')

# # print(hourly_range)

# # 2. Get unique locations (you might want to round coordinates to a grid)
# locations = dfd[['BEGIN_LAT', 'BEGIN_LON']].drop_duplicates()

# # 3. Create a full cartesian product of all hours × all locations
# from itertools import product
# hourly_grid = pd.DataFrame(
#     list(product(hourly_range, locations.itertuples(index=False))),
#     columns=['datetime_utc_naive', 'location']
# )

# print(hourly_grid)

# hourly_grid[['BEGIN_LAT', 'BEGIN_LON']] = pd.DataFrame(
#     hourly_grid['location'].tolist(), index=hourly_grid.index
# )

# # 4. Mark which hours had extreme weather events
# # hourly_grid['has_extreme_weather'] = 0
# # Merge with your events to flag extreme weather hours
# hourly_grid = hourly_grid.merge(dfd, on=['BEGIN_LAT', 'BEGIN_LON', 'datetime_utc_naive'], how='outer')

# # Sort hourly_grid by datetime_utc_naive and location
# hourly_grid = hourly_grid.sort_values(by=['datetime_utc_naive', 'location']).reset_index(drop=True)

# # print(hourly_grid)

### Create Grid Points

Create grid points for NOAA data based of datetime, latitude and longitude. The location bounds are set to match the data from ERA5 with increments of 0.25 degree.

The location bounds are:
- Latitude  : (26.0, 36.5)
- Longitude : (-106.75, -94.0)

In [ ]:
from itertools import product
import numpy as np

# 1. Define your time range (same as before)
start_date = dfd['datetime_utc_naive'].min().floor('H')-pd.Timedelta(hours=6)
end_date = dfd['datetime_utc_naive'].max().ceil('H')+pd.Timedelta(hours=4)
hourly_range = pd.date_range(start=start_date, end=end_date, freq='H')

print(start_date, end_date)

# 2. Instead of getting unique locations from dfd, create FULL grid
# Get the min/max bounds from your data
# lat_min = dfd['BEGIN_LAT'].min()
# lat_max = dfd['BEGIN_LAT'].max()
# lon_min = dfd['BEGIN_LON'].min()
# lon_max = dfd['BEGIN_LON'].max()

# Or you can manually set bounds for your region, e.g.:
lat_min, lat_max = 26.0, 36.5  # Texas region
lon_min, lon_max = -106.75, -94.0

# Create all lat/lon combinations with 0.25 degree increments
lats = np.arange(lat_min, lat_max + 0.25, 0.25)
lons = np.arange(lon_min, lon_max + 0.25, 0.25)

# Round to avoid floating point precision issues
lats = np.round(lats, 2)
lons = np.round(lons, 2)

# Create all location combinations
all_locations = list(product(lats, lons))

print(f"Time steps: {len(hourly_range)}")
print(f"Latitude points: {len(lats)} (from {lat_min} to {lat_max})")
print(f"Longitude points: {len(lons)} (from {lon_min} to {lon_max})")
print(f"Total locations: {len(all_locations)}")
print(f"Total grid points: {len(hourly_range) * len(all_locations)}")

# 3. Create full cartesian product of all hours × all locations
hourly_grid = pd.DataFrame(
    list(product(hourly_range, all_locations)),
    columns=['datetime_utc_naive', 'location']
)

# 4. Split the location tuple into separate columns
hourly_grid[['BEGIN_LAT', 'BEGIN_LON']] = pd.DataFrame(
    hourly_grid['location'].tolist(), index=hourly_grid.index
)

# Drop the location tuple column if not needed
hourly_grid = hourly_grid.drop('location', axis=1)

print(f"\nFinal grid shape: {hourly_grid.shape}")

# 5. Merge with your events to flag extreme weather hours
hourly_grid = hourly_grid.merge(
    dfd, 
    on=['BEGIN_LAT', 'BEGIN_LON', 'datetime_utc_naive'], 
    how='left'  # Use 'left' to keep all grid points
)

# # 6. Sort by datetime and location
# hourly_grid = hourly_grid.sort_values(
#     by=['datetime_utc_naive', 'BEGIN_LAT', 'BEGIN_LON']
# ).reset_index(drop=True)

print(f"\nFinal grid shape: {hourly_grid.shape}")
# print(hourly_grid.head())

In [ ]:
# # Check for duplicate location-time combinations in dfd
# duplicates = dfd.groupby(['BEGIN_LAT', 'BEGIN_LON', 'datetime_utc_naive']).size()
# print(f"Max events at same location/time: {duplicates.max()}")
# print(f"Number of location-time combos with multiple events: {(duplicates > 1).sum()}")

# print(duplicates)

Replace the NaN data with:
- has_extreme_weather --> False     (there's no extreme weather event)
- EVENT_TYPE --> NA                 (there's no extreme weather event)
- STATE --> Texas

In [ ]:
# Convert NaN to False
hourly_grid['has_extreme_weather'] = hourly_grid['has_extreme_weather'].fillna(False)

# Convert NaN to 'NA'
hourly_grid['EVENT_TYPE'] = hourly_grid['EVENT_TYPE'].fillna('NA')

# Update 'STATE' column in hourly_grid to "Texas" for all rows
hourly_grid['STATE'] = 'Texas'

# Fill Nan in END_LAT and END_LON
hourly_grid['END_LAT'] = hourly_grid['END_LAT'].fillna(hourly_grid['BEGIN_LAT'])
hourly_grid['END_LON'] = hourly_grid['END_LON'].fillna(hourly_grid['BEGIN_LON'])

hourly_grid.head()

In [ ]:
for col in hourly_grid.columns:
    print(col)

print('--------------------------------')

for col in ['EVENT_TYPE', 'STATE', 'has_extreme_weather']:
    print(hourly_grid[col].value_counts(dropna=False))
    print("--------------------------------")


In [ ]:
# Export to CSV
hourly_grid.to_csv('./data/noaa_hourly_grid.csv', index=False)

## ERA5 Data

Based on the data, the datasets are separated into two groups based on the characteristic of how the data being generated. 
- dsA (dataset 0) has the dimension of (time: 5880, lat: 45, long: 53)
- dsB (dataset 1) has the dimension of (time: 491, step: 12 lat: 45, long: 53)

In [ ]:
import cfgrib
import xarray as xr

datasets = cfgrib.open_datasets('./data/era5/era5_2023_texas_new.grib')

# Get the first and second group of datasets
dsA = datasets[0]
dsB = datasets[1]

In [ ]:
# Sort both by time
dsA = dsA.sortby("time")
dsB = dsB.sortby("time")

# Convert time to pandas datetime
dsA['time'] = pd.to_datetime(dsA.time.values)
dsB['time'] = pd.to_datetime(dsB.time.values)

print(dsA.dims)
print(dsB.dims)

In [ ]:
lat_min = dsA['latitude'].min().item()
lat_max = dsA['latitude'].max().item()
lon_min = dsA['longitude'].min().item()
lon_max = dsA['longitude'].max().item()

print(lat_min, lat_max)
print(lon_min, lon_max)

### Combining Time and Step

This process combines both time and step in dsB into a new dimension, which make dsB dimension into (time: 5892, lat: 45, long: 53). The new dimension (time) will replace the previous time and step dimension.

In [ ]:
print(dsB.time[0].values, dsB.time[-1].values)
print("================================================")

# Calculate valid_time for each (time, step) combination
valid_time = dsB["time"] + dsB["step"]

# Stack both the data and valid_time together
ds_stacked = dsB.stack(ts=("time", "step"))
valid_time_stacked = valid_time.stack(ts=("time", "step"))

# IMPORTANT: Drop the multi-index coordinates FIRST to avoid conflicts
ds_hourly = ds_stacked.reset_coords(drop=True)
ds_hourly = ds_hourly.drop_vars(['time', 'step'], errors='ignore')

# NOW rename the dimension and assign new coordinate values
ds_hourly = ds_hourly.rename({'ts': 'time'})
ds_hourly = ds_hourly.assign_coords(time=valid_time_stacked.values)

# Sort by time
ds_hourly = ds_hourly.sortby('time')

# Remove duplicate times if they exist
_, unique_indices = np.unique(ds_hourly['time'].values, return_index=True)
ds_hourly = ds_hourly.isel(time=sorted(unique_indices))

dsB = ds_hourly

print(dsB.time[0].values, dsB.time[-1].values)
print("================================================")

print(f"Final time shape: {dsB.time.shape}")
print(f"Final dimensions: {dict(dsB.dims)}")

In [ ]:
# Convert time to pandas datetime
dsA['time'] = pd.to_datetime(dsA.time.values)
dsB['time'] = pd.to_datetime(dsB.time.values)

# # IMPORTANT: Transpose dsB to match dsA's dimension order BEFORE reindexing
dsB = dsB.transpose('time', 'latitude', 'longitude')

# # Check time alignment
print(f"dsA time range: {dsA.time.values[0]} to {dsA.time.values[-1]}")
print(f"dsB time range: {dsB.time.values[0]} to {dsB.time.values[-1]}")
print(f"dsA shape: {dsA.dims}")
print(f"dsB shape: {dsB.dims}")

# Reindex dsB to match dsA's time coordinate
# Use a more lenient tolerance or interpolation
dsB_on_A = dsB.interp(time=dsA.time, method='nearest', kwargs={"fill_value": "extrapolate"})

# Alternative: If interpolation doesn't work, use reindex with no tolerance
# dsB_on_A = dsB.reindex(time=dsA.time, method='nearest')

# Verify that cp values are preserved
print(f"\nBefore merge - dsB cp non-NaN count: {np.sum(~np.isnan(dsB['tp'].values))}")
print(f"After reindex - dsB_on_A cp non-NaN count: {np.sum(~np.isnan(dsB_on_A['tp'].values))}")

# Merge with proper compatibility handling
ds = xr.merge([dsA, dsB_on_A], compat="no_conflicts")

# Verify merge result
print(f"After merge - ds tp non-NaN count: {np.sum(~np.isnan(ds['tp'].values))}")
print(ds.data_vars)
print(ds.dims)

print(f"ds Merge time range: {ds.time.values[0]} to {ds.time.values[-1]}")

#### Validate the values in dsB data_vars

Making sure that the value from dsB are not being replaced by NaN fater merging.

In [ ]:
tpVal = ds['tp'].values
mask_not_nan = ~np.isnan(tpVal)
tpVal_not_nan = tpVal[mask_not_nan]
print("tp values from dsB where not nan:", tpVal_not_nan.shape)

i10fgVal = ds['i10fg'].values
mask_not_nan = ~np.isnan(i10fgVal)
i10fgVal_not_nan = i10fgVal[mask_not_nan]
print("i10fg values from dsB where not nan:", i10fgVal_not_nan.shape)

In [ ]:
# Apply chunking for Dask
ds = ds.chunk({
    'time': 200,
    'latitude': -1,
    'longitude': -1
})

# Convert time to pandas datetime
# ds['time'] = pd.to_datetime(ds.time.values)

print("\n✅ Dataset loaded with all variables:")
print(f"Total variables: {len(ds.data_vars)}")
print(f"Variables: {sorted(list(ds.data_vars.keys()))}")
print(f"\nDataset info:")
print(ds)

In [ ]:
# # Print a sample of 5 rows of the dataset for all variables
# print("\n📝 Sample 5 rows from the dataset (all variables):")
# print(ds.to_dataframe().reset_index().head(5))

# Merging NOAA and ERA5 Datasets

## Without Time Window



In [ ]:
print(f"Grid shape: {hourly_grid.shape}")

# Extract coordinates as numpy arrays
times = hourly_grid['datetime_utc_naive'].values
lats = hourly_grid['BEGIN_LAT'].values
lons = hourly_grid['BEGIN_LON'].values

print(f"Total points to extract: {len(times)}")
print(f"Unique times: {len(np.unique(times))}")
print(f"Unique lats: {len(np.unique(lats))}")
print(f"Unique lons: {len(np.unique(lons))}")

In [ ]:
# Convert ds times to numpy array
ds_times = ds.time.values  # This gives numpy datetime64 array
ds_lats = ds.latitude.values
ds_lons = ds.longitude.values

# Vectorized nearest neighbor search for numeric values
def find_nearest_indices_numeric(query_vals, reference_vals):
    """Find nearest index for each query value in reference array"""
    distances = np.abs(query_vals[:, np.newaxis] - reference_vals[np.newaxis, :])
    nearest_idx = np.argmin(distances, axis=1)
    return nearest_idx

# For datetime: convert to int64 (nanoseconds since epoch)
def find_nearest_indices_datetime_chunked(query_times, reference_times, chunk_size=100000):
    """Find nearest index for datetime arrays, processing in chunks"""
    # Convert to int64 (nanoseconds)
    query_ns = query_times.astype('datetime64[ns]').astype(np.int64)
    ref_ns = reference_times.astype('datetime64[ns]').astype(np.int64)
    
    nearest_idx = np.zeros(len(query_ns), dtype=np.int64)
    
    for start in range(0, len(query_ns), chunk_size):
        end = min(start + chunk_size, len(query_ns))
        chunk = query_ns[start:end]
        distances = np.abs(chunk[:, np.newaxis] - ref_ns[np.newaxis, :])
        nearest_idx[start:end] = np.argmin(distances, axis=1)
        
        if (start // chunk_size + 1) % 5 == 0:
            print(f"Processed {end}/{len(query_ns)} time lookups")
    
    return nearest_idx

# Find indices for all points at once
# For times: use the datetime-specific function
time_indices = find_nearest_indices_datetime_chunked(times, ds_times)

# For lat/lon: use the numeric function
lat_indices = find_nearest_indices_numeric(lats, ds_lats)
lon_indices = find_nearest_indices_numeric(lons, ds_lons)

print(f"Time indices range: {time_indices.min()} to {time_indices.max()}")
print(f"Lat indices range: {lat_indices.min()} to {lat_indices.max()}")
print(f"Lon indices range: {lon_indices.min()} to {lon_indices.max()}")

In [ ]:
# Convert to torch tensors for GPU acceleration
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Define batch size to avoid memory issues
BATCH_SIZE = 50000  # Adjust based on available RAM

# Initialize dictionary to store results
era5_data = {var: np.full(len(hourly_grid), np.nan) for var in ds.data_vars}

# Process in batches
n_batches = (len(hourly_grid) + BATCH_SIZE - 1) // BATCH_SIZE

# Process each variable
for var in ds.data_vars:
    print(f"Processing {var}...")
    
    # Load ERA5 data for this variable to GPU
    era5_tensor = torch.from_numpy(ds[var].values).to(device)
    
    # Create index tensors
    time_idx_tensor = torch.from_numpy(time_indices).long().to(device)
    lat_idx_tensor = torch.from_numpy(lat_indices).long().to(device)
    lon_idx_tensor = torch.from_numpy(lon_indices).long().to(device)
    
    # Batch extraction to avoid memory issues
    batch_results = []
    for batch_idx in range(n_batches):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min((batch_idx + 1) * BATCH_SIZE, len(hourly_grid))
        
        batch_time = time_idx_tensor[start_idx:end_idx]
        batch_lat = lat_idx_tensor[start_idx:end_idx]
        batch_lon = lon_idx_tensor[start_idx:end_idx]
        
        # Extract values using advanced indexing
        batch_values = era5_tensor[batch_time, batch_lat, batch_lon]
        batch_results.append(batch_values.cpu().numpy())
    
    # Concatenate and add to hourly_grid
    hourly_grid[var] = np.concatenate(batch_results)
    
print("All variables added to hourly_grid!")

# Copy hourly_grid to a new variable
merged_df = hourly_grid.copy()




In [ ]:
# merged_df.head(20)

# # get data from merged_df where datetime_utc_naive is 31 May 2023 20:00
# selected_time = pd.Timestamp('2023-05-31 20:00:00')
# filtered_df = merged_df[(merged_df['datetime_utc_naive'] == selected_time) & (merged_df['BEGIN_LAT'] == 35.0) & (merged_df['BEGIN_LON'] == -102.0)]

# filtered_df.head(20)

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

HOURS_BEFORE = 6
HOURS_AFTER = 4

def format_row_as_text(row, timestamp_col='datetime_utc_naive', hours_before=HOURS_BEFORE, hours_after=HOURS_AFTER):
    obs_before = ""
    obs_after = ""
    ground_truth = {}
    obs_before_dict = {}

    lat = row['BEGIN_LAT']
    lon = row['BEGIN_LON']

    # Convert to string first to avoid initial float errors
    u10 = Decimal(str(row.get('u10', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    v10 = Decimal(str(row.get('v10', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    u100 = Decimal(str(row.get('u100', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    v100 = Decimal(str(row.get('v100', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    tp = Decimal(str(row.get('tp', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    sst = Decimal(str(row.get('sst', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    msl = Decimal(str(row.get('msl', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    i10fg = Decimal(str(row.get('i10fg', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    t2m = Decimal(str(row.get('t2m', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    d2m = Decimal(str(row.get('d2m', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
    sp = Decimal(str(row.get('sp', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)

    current = (
        f"\nAt {row['datetime_utc_naive']}, at the location ({lat}, {lon}), the weather observations were recorded as follows: "
        f"The weather event is: {row.get('EVENT_TYPE', 'unknown')}, "
        f"Eastward wind speed at 10 meters above ground: {row.get('u10', 0)} m/s, "
        f"Northward wind speed at 10 meters above ground: {row.get('v10', 0)} m/s, "
        f"Eastward wind speed at 100 meters above ground: {row.get('u100', 0)} m/s, "
        f"Northward wind speed at 100 meters above ground: {row.get('v100', 0)} m/s, "
        f"Mean sea level pressure: {row.get('msl', 0)} Pa, "
        f"Instantaneous 10 metre wind gust: {row.get('i10fg', 0)} m/s, "
        f"Sea surface temperature: {row.get('sst', 0)} K, "
        f"2 meters temperature: {row.get('t2m', 0)} K, "
        f"2 meters dewpoint temperature: {row.get('d2m', 0)} K, "
        f"Total precipitation: {row.get('tp', 0)} mm, "
        f"Surface pressure: {row.get('sp', 0)} Pa, "
    )

    # print(current)

    for i in range(HOURS_BEFORE, 0, -1):

        target_time = row['datetime_utc_naive'] - pd.Timedelta(hours=i)

        # Filter merged_df based on target_time, BEGIN_LAT, BEGIN_LON
        filtered_data = merged_df[(merged_df['datetime_utc_naive'] == target_time) & (merged_df['BEGIN_LAT'] == lat) & (merged_df['BEGIN_LON'] == lon)]

        data = filtered_data.iloc[0]

        # Convert to string first to avoid initial float errors
        u10 = Decimal(str(data.get('u10', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        v10 = Decimal(str(data.get('v10', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        u100 = Decimal(str(data.get('u100', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        v100 = Decimal(str(data.get('v100', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        tp = Decimal(str(data.get('tp', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        sst = Decimal(str(data.get('sst', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        msl = Decimal(str(data.get('msl', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        i10fg = Decimal(str(data.get('i10fg', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        t2m = Decimal(str(data.get('t2m', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        d2m = Decimal(str(data.get('d2m', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        sp = Decimal(str(data.get('sp', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        
        
        time_str = target_time.strftime('%Y-%m-%d %H:%M:%S')

        observation = (
            f"\nAt {time_str}, at the location ({lat}, {lon}), the weather observations were recorded as follows: "
            f"Eastward wind speed at 10 meters above ground: {u10} m/s, "
            f"Northward wind speed at 10 meters above ground: {v10} m/s, "
            f"Eastward wind speed at 100 meters above ground: {u100} m/s, "
            f"Northward wind speed at 100 meters above ground: {v100} m/s, "
            f"Mean sea level pressure: {msl} Pa, "
            f"Instantaneous 10 metre wind gust: {i10fg} m/s, "
            f"Sea surface temperature: {sst} K, "
            f"2 meters temperature: {t2m} K, "
            f"2 meters dewpoint temperature: {d2m} K, "
            f"Total precipitation: {tp} mm, "
            f"Surface pressure: {sp} Pa, "
        )

        obs_before_dict[time_str] = {
            "u10": u10,
            "v10": v10,
            "u100": u100,
            "v100": v100,
            "msl": msl,
            "i10fg": i10fg,
            "sst": sst,
            "t2m": t2m,
            "d2m": d2m,
            "tp": tp,
            "sp": sp,
        }
        
        obs_before += observation
    
    for i in range(1, HOURS_AFTER + 1):
        target_time = row['datetime_utc_naive'] + pd.Timedelta(hours=i)

        # Filter merged_df based on target_time, BEGIN_LAT, BEGIN_LON
        filtered_data = merged_df[(merged_df['datetime_utc_naive'] == target_time) & (merged_df['BEGIN_LAT'] == lat) & (merged_df['BEGIN_LON'] == lon)]

        data = filtered_data.iloc[0]
        
        # Convert to string first to avoid initial float errors
        u10 = Decimal(str(data.get('u10', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        v10 = Decimal(str(data.get('v10', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        u100 = Decimal(str(data.get('u100', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        v100 = Decimal(str(data.get('v100', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        tp = Decimal(str(data.get('tp', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        sst = Decimal(str(data.get('sst', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        msl = Decimal(str(data.get('msl', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        i10fg = Decimal(str(data.get('i10fg', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        t2m = Decimal(str(data.get('t2m', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        d2m = Decimal(str(data.get('d2m', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        sp = Decimal(str(data.get('sp', 0))).quantize(Decimal('0.000'), rounding=ROUND_HALF_UP)
        
        
        time_str = target_time.strftime('%Y-%m-%d %H:%M:%S')

        observation = (
            f"\nAt {time_str}, at the location ({lat}, {lon}), the weather observations were recorded as follows: "
            f"Eastward wind speed at 10 meters above ground: {u10} m/s, "
            f"Northward wind speed at 10 meters above ground: {v10} m/s, "
            f"Eastward wind speed at 100 meters above ground: {u100} m/s, "
            f"Northward wind speed at 100 meters above ground: {v100} m/s, "
            f"Mean sea level pressure: {msl} Pa, "
            f"Instantaneous 10 metre wind gust: {i10fg} m/s, "
            f"Sea surface temperature: {sst} K, "
            f"2 meters temperature: {t2m} K, "
            f"2 meters dewpoint temperature: {d2m} K, "
            f"Total precipitation: {tp} mm, "
            f"Surface pressure: {sp} Pa, "
        )

        ground_truth[time_str] = {
            "u10": u10,
            "v10": v10,
            "u100": u100,
            "v100": v100,
            "msl": msl,
            "i10fg": i10fg,
            "sst": sst,
            "t2m": t2m,
            "d2m": d2m,
            "tp": tp,
            "sp": sp,
        }
        
        obs_after += observation

    question = f"Based on the weather observations in the previous {HOURS_BEFORE} hours, will there be an extreme weather event in the next {HOURS_AFTER} hours?"

    # print(obs_before)
    
    return current, obs_before, obs_after, obs_before_dict, ground_truth, question

# Filter merged_df for June 2023 and has_extreme_weather
extreme_weather_df = merged_df[(merged_df['datetime_utc_naive'].dt.strftime('%Y-%m') == '2023-06') & merged_df['has_extreme_weather']]

# print(june_df)
row = extreme_weather_df.iloc[0]
# print(row)

extreme_weather, obs_before, obs_after, obs_before_dict, ground_truth, question = format_row_as_text(row, hours_before=HOURS_BEFORE, hours_after=HOURS_AFTER)



# print(row['datetime_utc_naive'])
print(f"Observations Before: {obs_before}")
print("--------------------------------\n")
print(f"Current Extreme Weather Event: {extreme_weather}")
print("--------------------------------\n")
print(f"Observations After: {obs_after}")
print("--------------------------------\n")
print(f"Observations Before JSON:\n {obs_before_dict}")
print("--------------------------------\n")
print(f"Ground Truth JSON:\n {ground_truth}")






In [ ]:
# Export the merged dataframe to CSV
merged_df.to_csv("./data/noaa_era5_merged.csv", index=False)



In [ ]:
import os

# Get the file size of the csv file in MB
file_size = os.path.getsize("./data/noaa_era5_merged.csv") / (1024 * 1024)
print(f"File size: {file_size} MB")

## With Time Window

In [ ]:
# from tqdm import tqdm
# import numpy as np
# import pandas as pd

# def extract_era5_with_time_window(event_row, ds, hours_before=0, hours_after=0):
#     """
#     Extract ERA5 data for a single event with a time window.
    
#     Parameters:
#     -----------
#     event_row : pandas Series
#         Row from NOAA dataset containing event information
#     ds : xarray Dataset
#         ERA5 dataset
#     hours_before : int
#         Number of hours before the event to include
#     hours_after : int
#         Number of hours after the event to include
    
#     Returns:
#     --------
#     dict : Dictionary containing ERA5 variables with time series data
#            Each variable will be a list of values across the time window
#     """
    
#     try:
#         # Get bounding box
#         lat_min = min(event_row['BEGIN_LAT'], event_row['END_LAT'])
#         lat_max = max(event_row['BEGIN_LAT'], event_row['END_LAT'])
#         lon_min = min(event_row['BEGIN_LON'], event_row['END_LON'])
#         lon_max = max(event_row['BEGIN_LON'], event_row['END_LON'])
        
#         # Check if it's a point event
#         is_point = (lat_min == lat_max) and (lon_min == lon_max)
        
#         # Calculate time window
#         event_time = pd.Timestamp(event_row['datetime_utc_naive'])
#         time_start = event_time - pd.Timedelta(hours=hours_before)
#         time_end = event_time + pd.Timedelta(hours=hours_after)

#         # print(time_start, time_end)
        
#         if is_point:
#             # Single point selection with time window
#             subset = ds.sel(
#                 latitude=lat_min,
#                 longitude=lon_min,
#                 time=slice(time_start, time_end),
#                 # method='nearest'
#             )
#         else:
#             # print("Area event")
#             # print(lat_max, lat_min, lon_min, lon_max)
            
#             # Area event - select region and average spatially
#             subset = ds.sel(
#                 latitude=slice(lat_max, lat_min),
#                 longitude=slice(lon_min, lon_max),
#                 time=slice(time_start, time_end)
#             )
#             # Average over the spatial dimensions
#             subset = subset.mean(dim=['latitude', 'longitude'])
        
#         # Compute to load data into memory
#         subset = subset.compute()
        
#         # Extract values into a dict with time dimension preserved
#         result = {}
        
#         # Add time information
#         result['time_window'] = [pd.Timestamp(t).isoformat() for t in subset['time'].values]
#         result['time_window_size'] = len(subset['time'])
#         result['event_time'] = event_time.isoformat()
        
#         # Extract each variable as a time series
#         for var in ds.data_vars:
#             result[var] = subset[var].values.tolist()  # Convert to list for easier storage
        
#         return result
        
#     except Exception as e:
#         print(f"Error extracting data for event {event_row.get('EVENT_ID', 'unknown')}: {e}")
#         return None

In [ ]:
# from tqdm import tqdm
# import pandas as pd
# import json

# # Define your time window parameters
# HOURS_BEFORE = 6  # Get 6 hours of data before the event
# HOURS_AFTER = 4   # Get 3 hours of data after the event

# # Test on one event first
# print(f"Testing extraction with {HOURS_BEFORE} hours before and {HOURS_AFTER} hours after event...")
# test_row = dfd.iloc[0]
# test_result = extract_era5_with_time_window(test_row, ds, hours_before=HOURS_BEFORE, hours_after=HOURS_AFTER)
# print("Test extraction successful!")
# print(f"Time window size: {test_result['time_window_size']} hours")
# print(f"Time range: {test_result['time_window'][0]} to {test_result['time_window'][-1]}")
# print(f"Sample ERA5 variable shape (sp): {len(test_result['sp'])} time points")

# # Create unique time-location combinations
# print("\nIdentifying unique time-location combinations...")
# dfd['_location_key'] = dfd.apply(
#     lambda x: f"{x['datetime_utc_naive']}_{x['BEGIN_LAT']}_{x['BEGIN_LON']}_{x['END_LAT']}_{x['END_LON']}", 
#     axis=1
# )

# unique_locations = dfd[['datetime_utc_naive', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON', '_location_key']].drop_duplicates(subset='_location_key')
# print(f"Found {len(unique_locations)} unique time-location points (out of {len(dfd)} total events)")

# # Extract ERA5 data with time windows
# print(f"\nExtracting ERA5 data with time windows ({HOURS_BEFORE}h before, {HOURS_AFTER}h after)...")
# era5_cache = {}

# for idx, row in tqdm(unique_locations.iterrows(), total=len(unique_locations), desc="Extracting ERA5"):
#     key = row['_location_key']
#     era5_data = extract_era5_with_time_window(row, ds, hours_before=HOURS_BEFORE, hours_after=HOURS_AFTER)
#     era5_cache[key] = era5_data

# print(f"\nSuccessfully extracted {len([v for v in era5_cache.values() if v is not None])} locations")

# # Merge ERA5 data with NOAA events
# print("\nMerging ERA5 data with NOAA events...")

# # Option 1: Store as JSON strings (if you want to keep it in a dataframe)
# # dfd['era5_time_window'] = dfd['_location_key'].apply(
# #     lambda key: json.dumps(era5_cache.get(key, {})) if era5_cache.get(key) else None
# # )

# # Option 2: Store as separate columns for each variable (preserving time series)
# # This creates columns like 'cape_timeseries', 'sp_timeseries', etc.
# for var in ds.data_vars:
#     dfd[f'{var}_timeseries'] = dfd['_location_key'].apply(
#         lambda key: era5_cache.get(key, {}).get(var, None)
#     )

# # Also store metadata
# dfd['time_window'] = dfd['_location_key'].apply(
#     lambda key: era5_cache.get(key, {}).get('time_window', None)
# )
# dfd['time_window_size'] = dfd['_location_key'].apply(
#     lambda key: era5_cache.get(key, {}).get('time_window_size', None)
# )

# # Clean up temporary column
# dfd = dfd.drop(columns=['_location_key'])

# print("\n✅ Merge complete!")
# print(f"Final dataframe shape: {dfd.shape}")
# print(f"\nColumns in merged dataset:")
# print(dfd.columns.tolist())

In [ ]:
# print(dfd.columns.tolist())

# # Export the merged dataframe to CSV
# dfd.to_csv("./data/noaa_era5_merged.csv", index=False)

In [ ]:
# # Print the first row's datetime_utc_naive and its respective time_window
# row = dfd.iloc[0]
# # print(f"datetime_utc_naive: {row['datetime_utc_naive']}")
# # print(f"time_window: {row['time_window']}")

# def format_row_as_text(row, timestamp_col='datetime_utc_naive'):
#     try:
#         # time_str = pd.to_datetime(row[timestamp_col]).strftime('%Y-%m-%d %H:%M')

#         time_window = row['time_window_size']

#         u10 = [round(val, 3) for val in row.get('u10_timeseries', [])]
#         v10 = [round(val, 3) for val in row.get('v10_timeseries', [])]
#         u100 = [round(val, 3) for val in row.get('u100_timeseries', [])]
#         v100 = [round(val, 3) for val in row.get('v100_timeseries', [])]
#         tp = [round(val, 3) for val in row.get('tp_timeseries', [])]
#         sst = [round(val, 3) for val in row.get('sst_timeseries', [])]
#         msl = [round(val, 3) for val in row.get('msl_timeseries', [])]
#         t2m = [round(val, 3) for val in row.get('t2m_timeseries', [])]
#         d2m = [round(val, 3) for val in row.get('d2m_timeseries', [])]
#         i10fg = [round(val, 3) for val in row.get('i10fg_timeseries', [])]
        
#         obs_before = ""
#         obs_after = ""
#         ground_truth = {}
#         obs_json = {}
#         for i in range(time_window, 0, -1):
#             i = i - 1
#             time_point = pd.to_datetime(row['time_window'][i])
#             time_str = time_point.strftime('%Y-%m-%d %H:%M:%S')
            
#             observation = (
#                 f"At {time_str}, at the location ({row['BEGIN_LAT']}, {row['BEGIN_LON']}), the weather observations were recorded as follows: "
#                 f"Eastward wind speed at 10 meters above ground: {u10[i]} m/s, "
#                 f"Northward wind speed at 10 meters above ground: {v10[i]} m/s, "
#                 f"Eastward wind speed at 100 meters above ground: {u100[i]} m/s, "
#                 f"Northward wind speed at 100 meters above ground: {v100[i]} m/s, "
#                 # f"Convective available potential energy: {cape[i]} J/kg, "
#                 f"Mean sea level pressure: {msl[i]} Pa, "
#                 # f"Boundary layer height: {blh[i]} m, "
#                 # f"Convective precipitation: {cp[i]} m, "
#                 # f"Convective inhibition: {cin[i]} J/kg, "
#                 # f"Convective rain rate: {crr[i]} kg/sm^2, "
#                 # f"Surface pressure: {sp[i]} Pa, "
#                 f"Instantaneous 10 metre wind gust: {i10fg[i]} m/s, "
#                 # f"Forecast surface roughness: {fsr[i]} m, "
#                 # f"Friction velocity: {zust[i]} m/s. "
#                 f"Sea surface temperature: {sst[i]} K, "
#                 f"2 meters temperature: {t2m[i]} K, "
#                 f"2 meters dewpoint temperature: {d2m[i]} K, "
#                 f"Total precipitation: {tp[i]} mm, "
#             )
            
#             # Separate based on HOURS_BEFORE and HOURS_AFTER
#             # Assume row['datetime_utc_naive'] is the "prediction/reference" time (t0)
#             t0 = pd.to_datetime(row['datetime_utc_naive'])
#             if time_point <= t0:
#                 if 'obs_before' not in locals():
#                     obs_before = ""
#                 obs_before += observation

#                 time_str2 = time_point.strftime('%Y-%m-%d %H:%M:%S')

#                 obs_json[time_str2] = {
#                     "u10": u10[i],
#                     "v10": v10[i],
#                     "u100": u100[i],
#                     "v100": v100[i],
#                     # "cape": cape[i],
#                     "msl": msl[i],
#                     # "blh": blh[i],
#                     # "cp": cp[i],
#                     # "cin": cin[i],
#                     # "crr": crr[i],
#                     # "sp": sp[i],
#                     "i10fg": i10fg[i],
#                     # "fsr": fsr[i],
#                     # "zust": zust[i],
#                     "sst": sst[i],
#                     "t2m": t2m[i],
#                     "d2m": d2m[i],
#                     "tp": tp[i],
#                 }
#             elif time_point > t0:
#                 if 'obs_after' not in locals():
#                     obs_after = ""
#                 obs_after += observation

#                 time_str2 = time_point.strftime('%Y-%m-%d %H:%M:%S')

#                 ground_truth[time_str2] = {
#                     "u10": u10[i],
#                     "v10": v10[i],
#                     "u100": u100[i],
#                     "v100": v100[i],
#                     # "cape": cape[i],
#                     "msl": msl[i],
#                     # "blh": blh[i],
#                     # "cp": cp[i],
#                     # "cin": cin[i],
#                     # "crr": crr[i],
#                     # "sp": sp[i],
#                     "i10fg": i10fg[i],
#                     # "fsr": fsr[i],
#                     # "zust": zust[i],
#                     "sst": sst[i],
#                     "t2m": t2m[i],
#                     "d2m": d2m[i],
#                     "tp": tp[i],
#                 }
#             # else:
#             #     result = observation
#             #     ground_truth = {}


#             # observations += observation
#         # obs_before += f"\n\nPlease describe weather observation in the next {HOURS_AFTER} hours using the same format as in the question."
#         question = f"Based on the weather observations in the previous {HOURS_BEFORE} hours, will there be an extreme weather event in the next {HOURS_AFTER} hours?"

#         return obs_before, obs_after, ground_truth, obs_json, question
#     except Exception as e:
#         return f"[Error formatting row: {e}]"


# Get one sample data from merged_df in the month of June 2023
june_df = merged_df[merged_df['datetime_utc_naive'].dt.strftime('%Y-%m') == '2023-06']

row = june_df.iloc[0]
# print(row)

observations, result, ground_truth, obs_json, question = format_row_as_text(row)
observations, result, ground_truth, obs_json, question = format_row_as_text(row)


# print(row['datetime_utc_naive'])
# print(observations)
# print("--------------------------------")
# print(result)
# print("--------------------------------")
# print(ground_truth)
# print("--------------------------------") 
# print(obs_json)
# print("--------------------------------")
# print(question)






In [ ]:
# dfd = dfd.sort_values(by='datetime_utc_naive').reset_index(drop=True)

In [ ]:
# observation_dicts = {}
# for idx, row in dfd.iterrows():
#     observation, result, ground_truth, obs_json, question = format_row_as_text(row)
#     observation_dicts[idx] = {'observation': observation, 'result': result, 'ground_truth': ground_truth, 'obs_json': obs_json, 'question': question}

# import json

# # Save observations to a jsonl file
# with open('./data/observations/jun_dataset.jsonl', 'w') as f:
#     for idx, data in observation_dicts.items():
#         f.write(json.dumps(data) + '\n')





## Single Data Merge

In [ ]:
# from tqdm import tqdm
# import numpy as np

# def extract_era5_optimized(event_row, ds):
#     """
#     Extract ERA5 data for a single event with proper error handling.
#     Handles both point events and area events with spatial averaging.
#     """
#     # print(event_row)
    
#     try:
#         # Get bounding box
#         lat_min = min(event_row['BEGIN_LAT'], event_row['END_LAT'])
#         lat_max = max(event_row['BEGIN_LAT'], event_row['END_LAT'])
#         lon_min = min(event_row['BEGIN_LON'], event_row['END_LON'])
#         lon_max = max(event_row['BEGIN_LON'], event_row['END_LON'])
        
#         # Check if it's a point event
#         is_point = (lat_min == lat_max) and (lon_min == lon_max)
        
#         if is_point:
#             # Single point selection
#             subset = ds.sel(
#                 latitude=lat_min,
#                 longitude=lon_min,
#                 time=event_row['datetime_utc_naive'],
#                 method='nearest'
#             )
#         else:
#             # Area event - select region and average spatially
#             subset = ds.sel(
#                 latitude=slice(lat_max, lat_min),  # xarray uses high-to-low for descending coords
#                 longitude=slice(lon_min, lon_max),
#                 time=event_row['datetime_utc_naive'],
#                 # method='nearest'
#             )
#             # Average over the spatial dimensions
#             subset = subset.mean(dim=['latitude', 'longitude'])
        
#         # Compute to load data into memory (Dask triggers here)
#         subset = subset.compute()
        
#         # Extract values into a dict
#         result = {}
#         for var in ds.data_vars:
#             result[var] = float(subset[var].values)
        
#         return result
        
#     except Exception as e:
#         # print(event_row)
#         print(f"Error extracting data for event {event_row.get('EVENT_ID', 'unknown')}: {e}")
#         return None

# # Test on one event
# test_row = dfd.iloc[0]
# test_result = extract_era5_optimized(test_row, ds)
# print("Test extraction successful!")
# print(f"Sample ERA5 data: {test_result}")

In [ ]:
# from tqdm import tqdm
# import pandas as pd

# # Create unique time-location combinations to avoid duplicate extractions
# print("Identifying unique time-location combinations...")
# dfd['_location_key'] = dfd.apply(
#     lambda x: f"{x['datetime_utc_naive']}_{x['BEGIN_LAT']}_{x['BEGIN_LON']}_{x['END_LAT']}_{x['END_LON']}", 
#     axis=1
# )

# unique_locations = dfd[['datetime_utc_naive', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON', '_location_key']].drop_duplicates(subset='_location_key')
# print(f"Found {len(unique_locations)} unique time-location points (out of {len(dfd)} total events)")

# # Extract ERA5 data for unique locations only
# print("\nExtracting ERA5 data for unique locations...")
# era5_cache = {}

# for idx, row in tqdm(unique_locations.iterrows(), total=len(unique_locations), desc="Extracting ERA5"):
#     key = row['_location_key']
#     era5_data = extract_era5_optimized(row, ds)
#     era5_cache[key] = era5_data

# print(f"\nSuccessfully extracted {len([v for v in era5_cache.values() if v is not None])} locations")

# # Step 3: Merge ERA5 data with NOAA events (fast dictionary lookups)
# print("\nMerging ERA5 data with NOAA events...")

# # Add ERA5 variables to the dataframe
# for var in ds.data_vars:
#     dfd[f'{var}'] = dfd['_location_key'].apply(
#         lambda key: era5_cache.get(key, {}).get(var, np.nan)
#     )

# # Clean up temporary column
# dfd = dfd.drop(columns=['_location_key'])

# print("\n✅ Merge complete!")
# print(f"Final dataframe shape: {dfd.shape}")
# print(f"\nColumns in merged dataset:")
# print(dfd.columns.tolist())

<div class="alert alert-block alert-info">
<b>Tip:</b> Use blue boxes (alert-info) for tips and notes. 
If it’s a note, you don’t have to include the word “Note”.
</div>

***

In [ ]:
# print(dfd.head(20))